In [1]:
# 1. Дані й параметри досліду
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CSV = "C:/Users/admin/source/repos/Kruskal/results/results_fr-par.csv"
MIN_EDGES = 1_000_000    
T_A = 8192                # поріг варіанта A (і найкращий поріг послідовного FK)

PARAMS = ["threads", "threshold", "partition_block", "filter_block", "par_cutoff", "psort_cutoff"]

def experiment_params(df):
    rows = {}
    for alg, d in df.groupby("algorithm", sort=False):
        row = {}
        for c in PARAMS:
            vals = sorted(d[c].dropna().unique())
            row[c] = ", ".join(str(int(v)) for v in vals) if vals else "-"
        repeats = d.groupby(["graph"] + PARAMS, dropna=False)["repeat"].nunique()
        row["повторів"] = f"{repeats.min()}–{repeats.max()}" if repeats.min() != repeats.max() else str(repeats.min())
        rows[alg] = row
    graphs = df["graph"].unique()
    print(f"Графи ({len(graphs)}): {', '.join(graphs)}")
    return pd.DataFrame(rows).T

df = pd.read_csv(CSV)
order = list(df["graph"].unique())
info = df.groupby("graph", sort=False)[["family", "V", "E"]].first()
big = df[df["E"] >= MIN_EDGES]
big_order = [g for g in order if info.loc[g, "E"] >= MIN_EDGES]

display(experiment_params(df))      # функція з клітинки «Параметри досліду»

geo = lambda r: np.exp(np.log(r).mean())


def selection(t, by):
    return (t.groupby(by)["rel"]
              .agg(середнє_відставання=geo, найгірший_граф="max", перемог=lambda r: int((r == 1.0).sum()))
              .sort_values("середнє_відставання"))


def noise(d, by):
    return d.groupby(["graph"] + by)["total_ms"].agg(lambda x: x.std() / x.mean()).median() * 100

Графи (19): B1, B2, B3, B4, B5, B6, B7, B8, B9, B10, B11, G1, C1, C2, C3, C4, C5, C6, C7


,threads,threshold,partition_block,filter_block,par_cutoff,psort_cutoff,повторів
fk,1,"8192, 65536, 262144, 1048576",-,-,-,-,6–25
fk-par,8,"8192, 65536, 262144, 1048576",8192,2048,65536,"0, 32768",7–25


In [2]:
info = df.groupby("graph", sort=False)[["family", "V", "E"]].first()
display(info)

,family,V,E
graph,,,
B1,dense,8000,16000
B2,dense,8000,32000
B3,dense,8000,64000
B4,dense,8000,128000
B5,dense,8000,256000
B6,dense,8000,512000
B7,dense,8000,1024000
B8,dense,8000,2048000
B9,dense,8000,4096000


In [3]:

fk = big[big["algorithm"] == "fk"].groupby(["graph", "threshold"], as_index=False)["total_ms"].mean()
fk["rel"] = fk["total_ms"] / fk.groupby("graph")["total_ms"].transform("min")

print("Послідовний FK, середній час, мс:")
display(fk.pivot(index="graph", columns="threshold", values="total_ms").reindex(big_order).round(1))
print("Послідовний FK: вибір порогу")
display(selection(fk, "threshold").round(3))
print(f"Типовий розкид між повторами: {noise(big[big['algorithm'] == 'fk'], ['threshold']):.1f}%")

base = fk[fk["threshold"] == T_A].set_index("graph")["total_ms"]   # еталон прискорення

Послідовний FK, середній час, мс:


threshold,8192,65536,262144,1048576
graph,,,,
B7,12.0,13.1,23.6,69.0
B8,24.0,25.3,30.2,76.5
B9,45.5,46.7,50.2,83.9
B10,94.8,95.5,98.7,137.9
B11,382.0,383.0,387.8,411.5
G1,1269.6,1178.7,1073.3,989.3
C4,64.9,63.3,62.7,67.1
C5,246.1,237.3,231.5,230.0
C6,1284.3,1222.6,1168.9,1118.6


Послідовний FK: вибір порогу


,середнє_відставання,найгірший_граф,перемог
threshold,,,
65536,1.058,1.191,0
8192,1.063,1.283,5
262144,1.131,1.968,1
1048576,1.497,5.750,4


Типовий розкид між повторами: 0.5%


In [4]:
#  Послідовний FK окремо по сімействах і два еталони
fk["family"] = fk["graph"].map(info["family"])
T_max = fk["threshold"].max()
for fam, d in fk.groupby("family"):
    s = selection(d, "threshold")
    print(f"=== {fam}: найкращий T = {s.index[0]}")
    display(s.round(3))

base_std = fk[fk["threshold"] == T_A].set_index("graph")["total_ms"]   #
best_fk = fk.loc[fk.groupby("graph")["total_ms"].idxmin()].set_index("graph")
base_best = best_fk["total_ms"]                                        # найкращий FK на кожному графі

=== dense: найкращий T = 8192


,середнє_відставання,найгірший_граф,перемог
threshold,,,
8192,1.000,1.000,5
65536,1.036,1.091,0
262144,1.236,1.968,0
1048576,2.211,5.750,0


=== grid: найкращий T = 1048576


,середнє_відставання,найгірший_граф,перемог
threshold,,,
1048576,1.000,1.000,1
262144,1.085,1.085,0
65536,1.191,1.191,0
8192,1.283,1.283,0


=== sparse: найкращий T = 1048576


,середнє_відставання,найгірший_граф,перемог
threshold,,,
1048576,1.017,1.070,3
262144,1.022,1.045,1
65536,1.056,1.093,0
8192,1.096,1.148,0


In [5]:
# 3. fk-par: час і прискорення для кожної пари (T, P)
par = big[big["algorithm"] == "fk-par"].copy()
par["psort_cutoff"] = par["psort_cutoff"].astype(int)
t = (par.groupby(["graph", "threshold", "psort_cutoff"], as_index=False)
        [["total_ms", "sorted_edges"]].mean())
t["sort"] = np.where(t["psort_cutoff"] == 0, "посл.", "паралельне")
t["family"] = t["graph"].map(info["family"])
tA = t[(t["threshold"] == T_A) & (t["psort_cutoff"] == 0)].set_index("graph")["total_ms"]

# При T = 8192 паралельне сортування не вмикається
wide = t.pivot(index="graph", columns=["threshold", "psort_cutoff"], values="total_ms").reindex(big_order)
t = t[~((t["threshold"] == T_A) & (t["psort_cutoff"] != 0))].copy()

t["speedup"] = t["graph"].map(base_best) / t["total_ms"]   # відносно найкращого послідовного FK на графі
t["vs_A"] = t["graph"].map(tA) / t["total_ms"]             # відносно варіанта A: > 1 — швидше
t["rel"] = t["total_ms"] / t.groupby("graph")["total_ms"].transform("min")


def table(values):
    return t.pivot(index="graph", columns=["threshold", "sort"], values=values).reindex(big_order)


print("Середній час fk-par, мс:")
display(table("total_ms").round(1))
print("Прискорення fk-par відносно найкращого послідовного FK на кожному графі:")
display(table("speedup").round(2))
print(f"Прискорення відносно варіанта A (fk-par, T = {T_A}, без паралельного сортування): > 1 — швидше")
display(table("vs_A").round(2))

Середній час fk-par, мс:


threshold 8192    65536              262144             1048576           
sort        посл.   посл. паралельне   посл. паралельне   посл. паралельне
graph                                                                     
B7            9.6    10.3        9.2    22.0       11.3    70.1       19.7
B8           17.2    18.3       18.0    29.1       19.8    77.4       30.0
B9           34.5    35.9       34.9    42.9       35.7    75.7       42.8
B10          74.0    75.2       74.2    81.9       74.2    95.7       77.1
B11         309.7   310.6      309.8   317.6      310.5   334.1      313.0
G1          817.0   726.9      653.8   714.2      591.4   718.8      554.4
C4           51.7    50.4       43.3    52.5       38.4    61.2       39.4
C5          187.3   174.6      157.2   181.1      140.3   184.8      138.3
C6          792.6   739.7      698.6   733.6      649.8   757.1      633.8
C7         3356.9  3115.5     3019.9  3070.3     2853.2  3104.6     2821.5

Прискорення fk-par відносно найкращого послідовного FK на кожному графі:


threshold 8192    65536              262144             1048576           
sort        посл.   посл. паралельне   посл. паралельне   посл. паралельне
graph                                                                     
B7           1.25    1.16       1.30    0.55       1.06    0.17       0.61
B8           1.40    1.31       1.34    0.82       1.21    0.31       0.80
B9           1.32    1.27       1.30    1.06       1.28    0.60       1.06
B10          1.28    1.26       1.28    1.16       1.28    0.99       1.23
B11          1.23    1.23       1.23    1.20       1.23    1.14       1.22
G1           1.21    1.36       1.51    1.39       1.67    1.38       1.78
C4           1.21    1.24       1.45    1.20       1.63    1.02       1.59
C5           1.23    1.32       1.46    1.27       1.64    1.24       1.66
C6           1.41    1.51       1.60    1.52       1.72    1.48       1.76
C7           1.50    1.62       1.67    1.64       1.77    1.63       1.79

Прискорення відносно варіанта A (fk-par, T = 8192, без паралельного сортування): > 1 — швидше


threshold 8192    65536              262144             1048576           
sort        посл.   посл. паралельне   посл. паралельне   посл. паралельне
graph                                                                     
B7            1.0    0.93       1.04    0.44       0.85    0.14       0.49
B8            1.0    0.94       0.96    0.59       0.87    0.22       0.57
B9            1.0    0.96       0.99    0.80       0.97    0.46       0.81
B10           1.0    0.98       1.00    0.90       1.00    0.77       0.96
B11           1.0    1.00       1.00    0.98       1.00    0.93       0.99
G1            1.0    1.12       1.25    1.14       1.38    1.14       1.47
C4            1.0    1.03       1.19    0.99       1.35    0.84       1.31
C5            1.0    1.07       1.19    1.03       1.34    1.01       1.35
C6            1.0    1.07       1.13    1.08       1.22    1.05       1.25
C7            1.0    1.08       1.11    1.09       1.18    1.08       1.19

In [6]:
# Вибір конфігурації fk-par: усі великі графи і окремо по сімействах
print("Усі великі графи:")
display(selection(t, ["threshold", "sort"]).round(3))
print(f"Типовий розкид між повторами: {noise(par, ['threshold', 'psort_cutoff']):.1f}%")

for fam, d in t.groupby("family"):
    s = selection(d, ["threshold", "sort"])
    print(f"\n=== {fam}: найкраща конфігурація T = {s.index[0][0]}, сортування {s.index[0][1]}")
    display(s.round(3))

Усі великі графи:


середнє_відставання  найгірший_граф  перемог
threshold sort                                                    
262144    паралельне                1.051           1.224        1
65536     паралельне                1.066           1.179        1
          посл.                     1.135           1.314        0
8192      посл.                     1.153           1.474        4
1048576   паралельне                1.175           2.145        4
262144    посл.                     1.324           2.387        0
1048576   посл.                     1.812           7.616        0

Типовий розкид між повторами: 1.4%

=== dense: найкраща конфігурація T = 8192, сортування посл.


середнє_відставання  найгірший_граф  перемог
threshold sort                                                    
8192      посл.                     1.008           1.041        4
65536     паралельне                1.012           1.047        1
          посл.                     1.048           1.121        0
262144    паралельне                1.079           1.224        0
1048576   паралельне                1.374           2.145        0
262144    посл.                     1.417           2.387        0
1048576   посл.                     2.536           7.616        0


=== grid: найкраща конфігурація T = 1048576, сортування паралельне


,,середнє_відставання,найгірший_граф,перемог
threshold,sort,,,
1048576,паралельне,1.000,1.000,1
262144,паралельне,1.067,1.067,0
65536,паралельне,1.179,1.179,0
262144,посл.,1.288,1.288,0
1048576,посл.,1.296,1.296,0
65536,посл.,1.311,1.311,0
8192,посл.,1.474,1.474,0



=== sparse: найкраща конфігурація T = 1048576, сортування паралельне


середнє_відставання  найгірший_граф  перемог
threshold sort                                                    
1048576   паралельне                1.006           1.026        3
262144    паралельне                1.013           1.025        1
65536     паралельне                1.109           1.136        0
          посл.                     1.209           1.314        0
262144    посл.                     1.225           1.367        0
8192      посл.                     1.284           1.354        0
1048576   посл.                     1.294           1.595        0

In [7]:
# Розкладання: ціна великого T і ефект паралельного сортування
big_T = sorted(T for T in t["threshold"].unique() if T > T_A)
P = max(t["psort_cutoff"].unique())

effect = pd.DataFrame(index=big_order)
for T in big_T:
    effect[f"T={T}: великий T"] = wide[(T_A, 0)] / wide[(T, 0)]    # < 1  великий T сам по собі сповільнює
    effect[f"T={T}: пар. сорт."] = wide[(T, 0)] / wide[(T, P)]     # > 1  паралельне сортування прискорює
print("Великий T: час варіанта A / час при цьому T з послідовним сортуванням")
print("Пар. сорт.: час із послідовним сортуванням / час із паралельним (при тому самому T)")
display(effect.round(2))

Великий T: час варіанта A / час при цьому T з послідовним сортуванням
Пар. сорт.: час із послідовним сортуванням / час із паралельним (при тому самому T)


,T=65536: великий T,T=65536: пар. сорт.,T=262144: великий T,T=262144: пар. сорт.,T=1048576: великий T,T=1048576: пар. сорт.
B7,0.93,1.12,0.44,1.95,0.14,3.55
B8,0.94,1.02,0.59,1.47,0.22,2.58
B9,0.96,1.03,0.80,1.20,0.46,1.77
B10,0.98,1.01,0.90,1.10,0.77,1.24
B11,1.00,1.00,0.98,1.02,0.93,1.07
G1,1.12,1.11,1.14,1.21,1.14,1.30
C4,1.03,1.16,0.99,1.37,0.84,1.55
C5,1.07,1.11,1.03,1.29,1.01,1.34
C6,1.07,1.06,1.08,1.13,1.05,1.19
C7,1.08,1.03,1.09,1.08,1.08,1.10


In [8]:
# %% 6. Підсумок: найкращий послідовний FK, варіант A і варіант B на кожному графі
T_B, P_B = 262144, 32768      # обрані параметри варіанта B

tB = t[(t["threshold"] == T_B) & (t["psort_cutoff"] == P_B)].set_index("graph")["total_ms"]
best_par = t.loc[t.groupby("graph")["total_ms"].idxmin()].set_index("graph")

# 1. Основна таблиця: прискорення відносно найкращого послідовного FK на кожному графі
final = pd.DataFrame({
    "FK: найкращий T": best_fk["threshold"],
    "FK, мс": base_best,
    "варіант A, мс": tA,
    "S варіанта A": base_best / tA,
    f"варіант B (T={T_B}), мс": tB,
    "S варіанта B": base_best / tB,
}).reindex(big_order)
print(f"Прискорення S - відносно найкращого послідовного FK на графі; p = 8, варіант B: T = {T_B}, P = {P_B}")
display(final.round(2))

# 2. Допоміжна таблиця: старий еталон і найкраща конфігурація fk-par на кожному графі
extra = pd.DataFrame({
    "FK (T=8192), мс": base_std,
    "S варіанта A відн. FK(8192)": base_std / tA,
    "S варіанта A відн. найкращого FK": base_best / tA,
    "fk-par: найкращий T": best_par["threshold"],
    "сортування": best_par["sort"],
    "S найкращого fk-par": base_best / best_par["total_ms"],
}).reindex(big_order)
print("Допоміжне: вплив вибору еталона і найкраща конфігурація fk-par, підібрана окремо для кожного графа")
display(extra.round(2))

Прискорення S - відносно найкращого послідовного FK на графі; p = 8, варіант B: T = 262144, P = 32768


,FK: найкращий T,"FK, мс","варіант A, мс",S варіанта A,"варіант B (T=262144), мс",S варіанта B
graph,,,,,,
B7,8192,12.00,9.58,1.25,11.27,1.06
B8,8192,24.03,17.19,1.40,19.81,1.21
B9,8192,45.53,34.47,1.32,35.66,1.28
B10,8192,94.76,74.04,1.28,74.18,1.28
B11,8192,381.96,309.74,1.23,310.45,1.23
G1,1048576,989.30,817.02,1.21,591.42,1.67
C4,262144,62.74,51.72,1.21,38.38,1.63
C5,1048576,230.00,187.27,1.23,140.26,1.64
C6,1048576,1118.61,792.62,1.41,649.82,1.72


Допоміжне: вплив вибору еталона і найкраща конфігурація fk-par, підібрана окремо для кожного графа


,"FK (T=8192), мс",S варіанта A відн. FK(8192),S варіанта A відн. найкращого FK,fk-par: найкращий T,сортування,S найкращого fk-par
graph,,,,,,
B7,12.00,1.25,1.25,65536,паралельне,1.30
B8,24.03,1.40,1.40,8192,посл.,1.40
B9,45.53,1.32,1.32,8192,посл.,1.32
B10,94.76,1.28,1.28,8192,посл.,1.28
B11,381.96,1.23,1.23,8192,посл.,1.23
G1,1269.55,1.55,1.21,1048576,паралельне,1.78
C4,64.92,1.26,1.21,262144,паралельне,1.63
C5,246.10,1.31,1.23,1048576,паралельне,1.66
C6,1284.28,1.62,1.41,1048576,паралельне,1.76


In [9]:
#  Підсумок: найкращий послідовний FK, варіант A і найкращий fk-par на кожному графі
best_par = t.loc[t.groupby("graph")["total_ms"].idxmin()].set_index("graph")
final = pd.DataFrame({
    "FK: найкращий T": best_fk["threshold"],
    "FK, мс": best_fk["total_ms"].round(2),
    "FK (T=8192), мс": base_std.round(2),
    "варіант A, мс": tA.round(2),
    "fk-par: найкращий T": best_par["threshold"],
    "сортування": best_par["sort"],
    "fk-par, мс": best_par["total_ms"].round(2),
}).reindex(big_order)
final["S варіанта A"] = (final["FK, мс"] / final["варіант A, мс"]).round(2)
final["S найкращого fk-par"] = (final["FK, мс"] / final["fk-par, мс"]).round(2)
final["S варіанта A відн. FK(8192)"] = (final["FK (T=8192), мс"] / final["варіант A, мс"]).round(2)
print("Прискорення S - відносно найкращого послідовного FK на графі (останній стовпець - відносно FK з T = 8192):")
display(final)

Прискорення S - відносно найкращого послідовного FK на графі (останній стовпець - відносно FK з T = 8192):


,FK: найкращий T,"FK, мс","FK (T=8192), мс","варіант A, мс",fk-par: найкращий T,сортування,"fk-par, мс",S варіанта A,S найкращого fk-par,S варіанта A відн. FK(8192)
graph,,,,,,,,,,
B7,8192,12.00,12.00,9.58,65536,паралельне,9.21,1.25,1.30,1.25
B8,8192,24.03,24.03,17.19,8192,посл.,17.19,1.40,1.40,1.40
B9,8192,45.53,45.53,34.47,8192,посл.,34.47,1.32,1.32,1.32
B10,8192,94.76,94.76,74.04,8192,посл.,74.04,1.28,1.28,1.28
B11,8192,381.96,381.96,309.74,8192,посл.,309.74,1.23,1.23,1.23
G1,1048576,989.30,1269.55,817.02,1048576,паралельне,554.44,1.21,1.78,1.55
C4,262144,62.74,64.92,51.72,262144,паралельне,38.38,1.21,1.63,1.26
C5,1048576,230.00,246.10,187.27,1048576,паралельне,138.33,1.23,1.66,1.31
C6,1048576,1118.61,1284.28,792.62,1048576,паралельне,633.79,1.41,1.76,1.62


In [10]:
#  Однаковий поріг: наскільки паралельна версія швидша за послідовну з тим самим T
fk_T = fk.set_index(["graph", "threshold"])["total_ms"]
t["speedup_same_T"] = fk_T.reindex(list(zip(t["graph"], t["threshold"]))).to_numpy() / t["total_ms"]
print("Прискорення fk-par відносно послідовного FK з тим самим порогом T:")
display(table("speedup_same_T").round(2))

Прискорення fk-par відносно послідовного FK з тим самим порогом T:


threshold 8192    65536              262144             1048576           
sort        посл.   посл. паралельне   посл. паралельне   посл. паралельне
graph                                                                     
B7           1.25    1.27       1.42    1.07       2.10    0.98       3.49
B8           1.40    1.38       1.41    1.04       1.52    0.99       2.55
B9           1.32    1.30       1.34    1.17       1.41    1.11       1.96
B10          1.28    1.27       1.29    1.21       1.33    1.44       1.79
B11          1.23    1.23       1.24    1.22       1.25    1.23       1.31
G1           1.55    1.62       1.80    1.50       1.81    1.38       1.78
C4           1.26    1.26       1.46    1.20       1.63    1.10       1.71
C5           1.31    1.36       1.51    1.28       1.65    1.24       1.66
C6           1.62    1.65       1.75    1.59       1.80    1.48       1.76
C7           1.71    1.77       1.82    1.71       1.84    1.63       1.79

In [11]:
# Скільки ребер сортується і перевірка шуму
sorted_pct = (t[t["psort_cutoff"] == 0]
                .pivot(index="graph", columns="threshold", values="sorted_edges").reindex(big_order)
                .div(info.loc[big_order, "E"], axis=0) * 100)
print("Відсортовано ребер, % від E:")
display(sorted_pct.round(2))

dup = (wide[(T_A, P)] / wide[(T_A, 0)] - 1) * 100
print(f"T = {T_A}: P = {P} і P = 0 мають збігатися (паралельне сортування не вмикається). Різниця, %:")
display(dup.round(1).to_frame("різниця, %").T)

Відсортовано ребер, % від E:


threshold,8192,65536,262144,1048576
graph,,,,
B7,1.14,3.50,24.23,100.00
B8,0.56,1.66,12.09,50.94
B9,0.29,1.44,5.14,20.16
B10,0.14,0.66,3.02,6.65
B11,0.03,0.15,0.63,2.06
G1,49.62,51.12,53.32,56.71
C4,13.09,15.46,17.94,24.04
C5,12.59,13.61,16.48,17.88
C6,12.18,12.89,13.88,16.08


T = 8192: P = 32768 і P = 0 мають збігатися (паралельне сортування не вмикається). Різниця, %:


graph,B7,B8,B9,B10,B11,G1,C4,C5,C6,C7
"різниця, %",2.4,1.8,-0.8,2.4,0.6,-0.5,-0.5,-0.9,0.6,-0.1


In [18]:
fam_table = t.pivot_table(index=["threshold", "sort"], columns="family", values="rel", aggfunc=geo)
fam_table["усі"] = t.groupby(["threshold", "sort"])["rel"].agg(geo)
display(fam_table.sort_values("усі").round(3))

family                dense   grid  sparse    усі
threshold sort                                   
262144    паралельне  1.079  1.067   1.013  1.051
65536     паралельне  1.012  1.179   1.109  1.066
          посл.       1.048  1.311   1.209  1.135
8192      посл.       1.008  1.474   1.284  1.153
1048576   паралельне  1.374  1.000   1.006  1.175
262144    посл.       1.417  1.288   1.225  1.324
1048576   посл.       2.536  1.296   1.294  1.812

In [19]:
fam_table = fam_table.rename(columns={"dense": "щільні", "grid": "решітка", "sparse": "розріджені", "усі": "усі графи"})
display(fam_table.sort_values("усі графи").round(3))

family                щільні  решітка  розріджені  усі графи
threshold sort                                              
262144    паралельне   1.079    1.067       1.013      1.051
65536     паралельне   1.012    1.179       1.109      1.066
          посл.        1.048    1.311       1.209      1.135
8192      посл.        1.008    1.474       1.284      1.153
1048576   паралельне   1.374    1.000       1.006      1.175
262144    посл.        1.417    1.288       1.225      1.324
1048576   посл.        2.536    1.296       1.294      1.812